# easy-capture — GPU PoC 검증 노트북

실제 MV/직캠 클립으로 **H1(추적 유지율)·H2(컷 재매칭)·H4(오디오 동기)** 를 GPU 에서 검증한다.

**사용법**
1. 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: GPU(T4)** 로 설정.
2. 셀을 위에서부터 순서대로 실행.
3. **짧은 클립(≤10초 권장)** 을 업로드.
4. 검출된 인물 그림을 보고 `TARGET` 인덱스를 고른 뒤 추적 셀 실행.

측정: 추적 유지율(AC-01 ≥80%), 컷 재매칭(AC-03 ≥70%), GPU 속도(AC-06 ≥10fps).

In [ ]:
# 1) 의존성 설치 + GPU 확인
!pip -q install -U "transformers>=4.55" accelerate "scenedetect[opencv]" av imageio imageio-ffmpeg
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("\u26a0 GPU 런타임이 아닙니다. 런타임 유형을 GPU 로 바꾸고 다시 실행하세요.")

In [ ]:
# 2) 클립 업로드 + 프레임 로드 (처리 길이 제한)
from google.colab import files
import cv2, numpy as np

MAX_SECONDS = 8  # 처리할 최대 길이(초)
print("짧은 MV/직캠 클립을 업로드하세요 (<=10초 권장).")
up = files.upload()
VIDEO = next(iter(up))

def load_frames_rgb(path, max_seconds):
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    maxf = int(fps * max_seconds)
    frames = []
    while len(frames) < maxf:
        ok, bgr = cap.read()
        if not ok:
            break
        frames.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames, fps

frames, FPS = load_frames_rgb(VIDEO, MAX_SECONDS)
H, W = frames[0].shape[:2]
print(f"{len(frames)} frames, {W}x{H}, {FPS:.1f}fps")

In [ ]:
# 3) 모델 로드 (SAM2 video + Grounding DINO)
from transformers import (Sam2VideoModel, Sam2VideoProcessor,
                          GroundingDinoForObjectDetection, AutoProcessor)
device = "cuda" if torch.cuda.is_available() else "cpu"
sam_proc = Sam2VideoProcessor.from_pretrained("facebook/sam2.1-hiera-tiny")
sam = Sam2VideoModel.from_pretrained("facebook/sam2.1-hiera-tiny").to(device).eval()
gd_proc = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-tiny")
gd = GroundingDinoForObjectDetection.from_pretrained("IDEA-Research/grounding-dino-tiny").to(device).eval()
print("loaded on", device)

In [ ]:
# 4) 0번 프레임에서 인물 자동 검출 → 후보 표시
from PIL import Image
from IPython.display import display

PROMPT = "person."   # 사물 추적 시 변경 (예: "car." / "dog.")

def detect(frame_rgb, text=PROMPT, thr=0.30, txt_thr=0.25):
    img = Image.fromarray(frame_rgb)
    inp = gd_proc(images=img, text=text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = gd(**inp)
    res = gd_proc.post_process_grounded_object_detection(
        out, threshold=thr, text_threshold=txt_thr,
        target_sizes=[(frame_rgb.shape[0], frame_rgb.shape[1])])[0]
    return res["boxes"].detach().cpu().numpy(), res["scores"].detach().cpu().numpy()

boxes, scores = detect(frames[0])
vis = frames[0].copy()
for i, (b, s) in enumerate(zip(boxes, scores)):
    x1, y1, x2, y2 = map(int, b)
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(vis, f"{i}:{s:.2f}", (x1, max(12, y1 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
display(Image.fromarray(vis))
print(f"검출 {len(boxes)}개. 다음 셀 TARGET 에 추적할 인덱스를 넣으세요.")

In [ ]:
# 5) 추적 대상 선택 (위 그림의 인덱스)
TARGET = 0

In [ ]:
# 6) 재매칭/유틸 함수
import time

def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0

def appearance(frame, box):
    x1, y1, x2, y2 = [int(max(0, v)) for v in box]
    crop = frame[y1:y2, x1:x2]
    if crop.size == 0:
        return np.zeros(48, np.float32)
    h = cv2.calcHist([crop], [0, 1, 2], None, [4, 4, 3],
                     [0, 256, 0, 256, 0, 256]).flatten()
    n = h.sum()
    return (h / n) if n > 0 else h

def cos(u, v):
    d = (np.linalg.norm(u) * np.linalg.norm(v)) or 1.0
    return float(np.dot(u, v) / d)

def rematch(prev_box, prev_app, cand_boxes, frame, w_pos=0.7, w_cls=0.3):
    best, score = -1, -1.0
    for i, b in enumerate(cand_boxes):
        s = w_pos * iou(prev_box, b) + w_cls * cos(prev_app, appearance(frame, b))
        if s > score:
            best, score = i, s
    return best, score

def mask_bbox(m):
    ys, xs = np.where(m)
    if len(xs) == 0:
        return None
    return [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]

In [ ]:
# 7) 컷 감지 + 컷마다 재매칭하여 구간별 SAM2 재추적 (H1+H2 통합)
from scenedetect import detect as sd_detect, ContentDetector

scenes = sd_detect(VIDEO, ContentDetector())
cut_frames = sorted({int(s.get_frames()) for s, _ in scenes
                     if 0 < int(s.get_frames()) < len(frames)})
print("컷 프레임:", cut_frames)

def track_segment(seg_frames, init_box):
    sess = sam_proc.init_video_session(video=seg_frames, inference_device=device, dtype=torch.float32)
    sam_proc.add_inputs_to_inference_session(
        sess, frame_idx=0, obj_ids=1, input_boxes=[[list(init_box)]])
    out = {}
    with torch.inference_mode():
        for o in sam.propagate_in_video_iterator(sess, start_frame_idx=0):
            m = sam_proc.post_process_masks([o.pred_masks], original_sizes=[(H, W)])[0]
            out[o.frame_idx] = (m.detach().cpu().numpy() > 0).squeeze()
    return out

bounds = [0] + cut_frames + [len(frames)]
masks_all, rematch_log = {}, []
last_box = [float(v) for v in boxes[TARGET]]
last_app = appearance(frames[0], last_box)

t = time.time()
for k in range(len(bounds) - 1):
    s, e = bounds[k], bounds[k + 1]
    if k == 0:
        ibox = last_box
    else:
        cb, cs = detect(frames[s])
        if len(cb) == 0:
            rematch_log.append((s, "검출0 → 갭")); continue
        bi, sc = rematch(last_box, last_app, cb, frames[s])
        rematch_log.append((s, f"score={sc:.2f} {'OK' if sc>=0.5 else '실패→갭'}"))
        if sc < 0.5:
            continue
        ibox = [float(v) for v in cb[bi]]
    for li, arr in track_segment(frames[s:e], ibox).items():
        gi = s + li
        masks_all[gi] = arr
        bb = mask_bbox(arr)
        if bb:
            last_box, last_app = bb, appearance(frames[gi], bb)
dt = time.time() - t

n = len(frames)
kept = sum(1 for a in masks_all.values() if a.sum() > 0)
print(f"\n=== 결과 ===")
print(f"추적 유지율(AC-01): {100*kept/n:.0f}%  ({kept}/{n} 프레임)")
print(f"GPU 처리 속도(AC-06): {n/dt:.1f} fps  (총 {dt:.1f}s)")
print("컷 재매칭(AC-03):")
for s, msg in rematch_log:
    print(f"  컷@frame{s}: {msg}")

In [ ]:
# 8) 추적 결과 오버레이 영상 생성 + 다운로드 (눈으로 품질 확인)
import imageio

overlay = []
for i, f in enumerate(frames):
    o = f.copy()
    a = masks_all.get(i)
    if a is not None and a.sum() > 0:
        o[a] = (0.45 * o[a] + 0.55 * np.array([255, 40, 40])).astype(np.uint8)
        bb = mask_bbox(a)
        if bb:
            cv2.rectangle(o, (bb[0], bb[1]), (bb[2], bb[3]), (255, 40, 40), 2)
    overlay.append(o)
imageio.mimsave("track_overlay.mp4", overlay, fps=FPS)
print("track_overlay.mp4 생성")
files.download("track_overlay.mp4")

In [ ]:
# 9) H4: 크롭 구간 + 원본 오디오 동기 MP4 (오디오 mux 검증)
import imageio_ffmpeg, subprocess

def export_clip(src, dst, start, end, cw, ch, cx=0, cy=0):
    ff = imageio_ffmpeg.get_ffmpeg_exe()
    cw -= cw % 2; ch -= ch % 2
    vf = f"crop={cw}:{ch}:{cx}:{cy},format=yuv420p"
    cmd = [ff, "-y", "-ss", str(start), "-to", str(end), "-i", src, "-vf", vf,
           "-c:v", "libx264", "-c:a", "aac",
           "-colorspace", "bt709", "-color_primaries", "bt709", "-color_trc", "bt709", dst]
    subprocess.run(cmd, check=True)

export_clip(VIDEO, "clip_audio.mp4", 1.0, min(4.0, MAX_SECONDS), 320, 320)
print("clip_audio.mp4 (오디오 포함) 생성")
files.download("clip_audio.mp4")

## 결과 해석

- **추적 유지율(AC-01)** ≥ 80%, **컷 재매칭(AC-03)** ≥ 70%, **GPU 속도(AC-06)** ≥ 10fps 이면 목표 충족.
- `track_overlay.mp4` 로 **실제로 최애를 잘 따라가는지·컷 후 다른 사람으로 안 바뀌는지** 눈으로 확인.
- 컷 재매칭 점수가 낮으면 `rematch` 의 `w_pos/w_cls` 가중치, threshold(0.5), DINO `PROMPT`/threshold 를 조정.
- 결과 수치를 `poc/REPORT.md` 의 미검증 항목에 채워 Go/No-Go 확정.